# Ingesta Batch - Farmia Lakehouse

Pipeline completo de ingesta batch para el proyecto **Farmia**. Ejecuta secuencialmente: limpieza del entorno, generación de datos sintéticos, ingesta mediante AutoLoader y validación de las tablas resultantes en la capa **Bronze**.

## 1. Reinicio del entorno Python
Se reinicia el intérprete para garantizar un estado limpio antes de ejecutar el pipeline.

In [0]:
%restart_python

## 2. Limpieza y preparación del almacenamiento
Se eliminan y recrean las carpetas base en Azure (`Landing`, `Raw`, `Bronze` y `Checkpoint`) para partir de un entorno limpio.

In [0]:
# Obtener las rutas base de las configuraciones de Spark
landing_path = spark.conf.get("spark.farmia.base_landing_path")
raw_path = spark.conf.get("spark.farmia.base_raw_path")
bronze_path = spark.conf.get("spark.farmia.base_bronze_path")
checkpoint_path = spark.conf.get("spark.farmia.base_checkpoint_path")

paths = [landing_path, raw_path, bronze_path, checkpoint_path]

# 1. Borrar carpetas de forma recursiva si existen
for path in paths:
    dbutils.fs.rm(path, recurse=True)

# 2. Recrear carpetas base
for path in paths:
    dbutils.fs.mkdirs(path)

print("Contenedor farmia-data limpiado y carpetas base recreadas:")
print(f"  - Landing:    {landing_path}")
print(f"  - Raw:        {raw_path}")
print(f"  - Bronze:     {bronze_path}")
print(f"  - Checkpoint: {checkpoint_path}")

# 3. Listar el contenido del contenedor en Azure
base_container = landing_path.rsplit("/", 1)[0]
display(dbutils.fs.ls(base_container))

## 3. Generación de datos sintéticos
Se ejecuta el script que simula las fuentes de datos del negocio Farmia y deposita los ficheros en la zona de **Landing**.

In [0]:
import scripts.generate_synthetic_data as gen

gen.main(mode="batch")

## 4. Ingesta Batch con AutoLoader
Se ejecutan secuencialmente las ingestas configuradas (`ecommerce_ventas`, `inventario_local`, `logistica_envios`, `meteorologia`) utilizando el motor de AutoLoader para llevar los datos de Landing a **Bronze**.

In [0]:
import src.batch.autoloader_engine as engine

# 1. Limpieza previa del esquema en hive_metastore
spark.sql("DROP DATABASE IF EXISTS hive_metastore.farmia_bronze CASCADE")
print("Esquema hive_metastore.farmia_bronze eliminado correctamente (si existía).")

# 2. Lista de configuraciones batch
batch_configs = [
    "ecommerce_ventas.json",
    "inventario_local.json",
    "logistica_envios.json",
    "meteorologia.json"
]

# 3. Ejecución secuencial de la ingesta
for config in batch_configs:
    print(f"\n==================================================")
    print(f" Lanzando ingesta para: {config}")
    print(f"==================================================")
    try:
        engine.run_batch_ingestion(config)
        print(f"Ingesta completada con éxito: {config}")
    except Exception as e:
        print(f"Error durante la ingesta de {config}: {e}")

## 5. Registro y validación de tablas Bronze
Se registran las tablas externas en `hive_metastore.farmia_bronze` y se visualiza una muestra de cada tabla de negocio para verificar la correcta ingesta.

In [0]:
# 1. Establecer el contexto en hive_metastore y la base de datos farmia_bronze
spark.sql("USE CATALOG hive_metastore")
spark.sql("CREATE DATABASE IF NOT EXISTS farmia_bronze")
spark.sql("USE farmia_bronze")

# 2. Eliminar la referencia a la tabla si quedó vacía previamente
spark.sql("DROP TABLE IF EXISTS ingestion_control")

# 3. Registrar la tabla externa apuntando a la ruta física Delta (que ya tiene los Parquets del streaming)
audit_path = f"{spark.conf.get('spark.farmia.base_bronze_path').rstrip('/')}/ingestion_control"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS ingestion_control
    USING DELTA
    LOCATION '{audit_path}'
""")
print(f"Tabla externa 'ingestion_control' vinculada correctamente en {audit_path}")

# 4. Listar todas las tablas registradas en el esquema
display(spark.sql("SHOW TABLES"))

# 5. Visualizar las 4 tablas de negocio
tables = [
    "farmia_ecommerce_ventas",
    "farmia_inventario_local",
    "farmia_logistica_envios",
    "farmia_meteorologia"
]

for table in tables:
    print(f"=== Tabla: {table} ===")
    display(spark.sql(f"SELECT * FROM {table} LIMIT 10"))

## 6. Auditoría de la ingesta
Consulta de la tabla `ingestion_control` para verificar la trazabilidad y el resultado de cada proceso de ingesta ejecutado.

In [0]:
%sql
USE CATALOG hive_metastore;
USE farmia_bronze;

SELECT * 
FROM ingestion_control
ORDER BY ingest_ts DESC;